# Preprocess

> Fill in a module description here

In [ ]:
#| default_exp preprocess

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from skimage.segmentation import find_boundaries, expand_labels
from imageio import imwrite
import cv2
from typing import Tuple
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import distance_transform_edt
import numpy as np
from skimage.color import rgb2hed, hed2rgb

In [ ]:
#| export
def clahe(
    image: np.ndarray,
    clip_limit: float,
    tile_grid_size: Tuple[int, int]
) -> np.ndarray:
    """
    Apply Contrast Limited Adaptive Histogram Equalization (CLAHE) to a grayscale image.

    Parameters
    ----------
    image : np.ndarray
        2D array representing the input grayscale image.
    clip_limit : float
        Threshold for contrast limiting. Higher values result in stronger contrast enhancement.
    tile_grid_size : tuple of int
        Size (height, width) of the grid for histogram equalization (e.g., (8, 8)).

    Returns
    -------
    np.ndarray
        CLAHE-enhanced image as a 2D array.
    """
    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid_size
    )
    return clahe.apply(image)



In [ ]:
#| export
def process_ihc_image(ihc_rgb: np.ndarray) -> tuple:
    """
    Process an IHC image by separating its stains.

    Parameters
    ----------
    ihc_rgb : numpy.ndarray
        The IHC image as a NumPy array in BGR format.

    Returns
    -------
    ihc_rgb : numpy.ndarray
        The original IHC image in RGB format.
    ihc_h : numpy.ndarray
        The hematoxylin stain image.
    ihc_e : numpy.ndarray
        The eosin stain image.
    ihc_d : numpy.ndarray
        The DAB stain image.
    """

    # Separate the stains from the IHC image
    ihc_hed = rgb2hed(ihc_rgb)

    # Create an RGB image for each of the stains
    null = np.zeros_like(ihc_hed[:, :, 0])
    ihc_h = hed2rgb(np.stack((ihc_hed[:, :, 0], null, null), axis=-1))
    ihc_e = hed2rgb(np.stack((null, ihc_hed[:, :, 1], null), axis=-1))
    ihc_d = hed2rgb(np.stack((null, null, ihc_hed[:, :, 2]), axis=-1))

    return ihc_rgb, ihc_h, ihc_e, ihc_d